In [6]:
# 1 — Load and inspect the Wandercraft dataset
import pandas as pd

wc = pd.read_csv("/content/wandercraft_finance_dataset_sample.csv", sep=";")

print(wc.shape)
print()
print(wc["doc_type"].value_counts())
print()
print(wc.dtypes)

wc.head()

(137812, 18)

doc_type
supplier_invoice    96682
bank_entry          24808
expense_report      16322
Name: count, dtype: int64

doc_id                   object
doc_type                 object
doc_date                 object
supplier_or_employee     object
country                  object
category                 object
account                   int64
currency                 object
amount_ht               float64
vat_rate                float64
vat_amount              float64
amount_ttc              float64
fx_rate                 float64
amount_eur              float64
payment_date             object
po_number                object
cost_center              object
status                   object
dtype: object


,doc_id,doc_type,doc_date,supplier_or_employee,country,category,account,currency,amount_ht,vat_rate,vat_amount,amount_ttc,fx_rate,amount_eur,payment_date,po_number,cost_center,status
0,INV-117538,supplier_invoice,2025-08-03,TUV Rheinland,DE,certification,6226,EUR,4915.60,0.2,983.12,5898.72,1.00,5898.72,2025-10-02,PO-292143,Clinical,paid
1,BNK-5020385,bank_entry,2024-12-19,FedEx Express,US,logistics,512000,EUR,NaN,NaN,NaN,18955.81,1.00,18955.81,2024-12-19,NaN,G&A,reconciled
2,INV-117848,supplier_invoice,2025-12-04,Keysight Technologies,US,instruments,6183,USD,8418.03,0.0,0.00,8418.03,0.92,7744.59,2026-01-03,PO-233316,Quality,paid
3,BNK-5014007,bank_entry,2023-05-23,Mouser Electronics,US,electronics,512000,EUR,NaN,NaN,NaN,41631.70,1.00,41631.70,2023-05-23,NaN,Operations,reconciled
4,EXP-705487,expense_report,2023-01-23,ROUX Camille,FR,Software subscription,6251,USD,2338.74,0.0,0.00,2338.74,0.92,2151.64,2023-02-13,NaN,Quality,paid


The dataset loads correctly: 137,812 rows across 18 columns, a realistic volume for an annual accounting file.

It combines three document types: ~96,700 supplier invoices, ~24,800 bank entries and ~16,300 expense reports.

Data types are as expected: dates and labels as text (object), all monetary fields as float64.

The HT/VAT columns contain missing values by design — bank entries carry no VAT breakdown — which is normal and will be handled in the controls.

In [7]:
# 2 — Inspect key columns before designing controls
print(wc["currency"].value_counts())
print()
print("Distinct VAT rates:", wc["vat_rate"].dropna().nunique())
print("Standard rate present:", 0.20 in wc["vat_rate"].values)
print()
print("Missing values per column:")
print(wc.isna().sum())
print()
print("Statuses:", wc["status"].unique())
print()
print("FX rate per currency:")
print(wc.groupby("currency")["fx_rate"].unique())

currency
EUR    111199
USD     20159
CHF      3230
GBP      3224
Name: count, dtype: int64

Distinct VAT rates: 359
Standard rate present: True

Missing values per column:
doc_id                      0
doc_type                    0
doc_date                    0
supplier_or_employee        0
country                     0
category                    0
account                     0
currency                    0
amount_ht               25018
vat_rate                24808
vat_amount              24808
amount_ttc                  0
fx_rate                     0
amount_eur                  0
payment_date                0
po_number               55565
cost_center                 0
status                      0
dtype: int64

Statuses: ['paid' 'reconciled']

FX rate per currency:
currency
CHF         [1.04]
EUR          [1.0]
GBP         [1.17]
USD    [0.92, 1.0]
Name: fx_rate, dtype: object


There are four currencies in the data: EUR appears 111,199 times, USD 20,159 times, CHF 3,230 times and GBP 3,224 times.

The vat_rate column contains 361 different values.

The columns with missing values are: amount_ht (25,018 missing), vat_rate (24,808 missing), vat_amount (24,808 missing) and po_number (55,565 missing). All other columns have no missing values.

There are two statuses in the data: "paid" and "reconciled".

Each currency uses a single FX rate — CHF 1.04, EUR 1.0, GBP 1.17 — except USD, which appears with two rates: 0.92 and 1.0.

In [8]:
# 3 — Define control rules (each returns a label and the flagged rows)
LEGAL_VAT_RATES = [0.0, 0.055, 0.10, 0.20]
EXPECTED_FX = {"EUR": 1.0, "USD": 0.92, "CHF": 1.04, "GBP": 1.17}
POLICY_LIMIT = 3000          # expense policy threshold (EUR)
LARGE_INVOICE = 30000        # invoice amount requiring a purchase order

invoices_and_expenses = wc["doc_type"].isin(["supplier_invoice", "expense_report"])

# Check that the invoiced total is arithmetically consistent: HT + VAT must equal TTC.
def ctrl_ttc_consistency(data, tol=0.01):
    mask = invoices_and_expenses & (
        (data["amount_ht"].fillna(0) + data["vat_amount"].fillna(0) - data["amount_ttc"]).abs() > tol
    )
    return "TTC equals HT + VAT", data[mask]

# Ensure the VAT rate is one of the legal French rates (0 / 5.5 / 10 / 20%), not an arbitrary value.
def ctrl_vat_rate_legal(data):
    mask = invoices_and_expenses & data["vat_rate"].notna() & ~data["vat_rate"].round(3).isin([round(r, 3) for r in LEGAL_VAT_RATES])
    return "VAT rate is a legal rate", data[mask]

# Flag invoices/expenses where the HT amount is missing (bank entries legitimately have none).
def ctrl_missing_amount(data):
    mask = invoices_and_expenses & data["amount_ht"].isna()
    return "Amount HT present on invoices/expenses", data[mask]

# Detect negative booked amounts, which should not occur outside of credit notes.
def ctrl_negative_amount(data):
    mask = data["amount_ttc"] < 0
    return "No negative amounts", data[mask]

# Verify chronology: a payment cannot be dated before the document it settles.
def ctrl_payment_after_invoice(data):
    mask = pd.to_datetime(data["payment_date"]) < pd.to_datetime(data["doc_date"])
    return "Payment date not before document date", data[mask]

# Check that the FX rate applied matches the expected rate for the document's currency.
def ctrl_fx_consistency(data):
    expected = data["currency"].map(EXPECTED_FX)
    mask = (data["fx_rate"] - expected).abs() > 0.001
    return "FX rate matches currency", data[mask]

# Confirm the EUR conversion is correct: amount_ttc x fx_rate must equal amount_eur.
def ctrl_eur_conversion(data, tol=0.5):
    mask = (data["amount_ttc"] * data["fx_rate"] - data["amount_eur"]).abs() > tol
    return "amount_eur equals amount_ttc x fx_rate", data[mask]

# Enforce the expense policy: employee expenses above the threshold require review.
def ctrl_expense_policy(data):
    mask = (data["doc_type"] == "expense_report") & (data["amount_ht"] > POLICY_LIMIT)
    return "Expense within policy limit", data[mask]

# Apply the "approved-to-pay" rule: large invoices must be backed by a purchase order.
def ctrl_large_invoice_has_po(data):
    mask = (data["doc_type"] == "supplier_invoice") & (data["amount_ttc"].abs() > LARGE_INVOICE) & (data["po_number"].fillna("") == "")
    return "Large invoice has a PO number", data[mask]

# Identify potential duplicate invoices: same supplier, amount and currency booked more than once.
def ctrl_duplicate_invoices(data):
    inv = data[data["doc_type"] == "supplier_invoice"]
    dup_mask = inv.duplicated(subset=["supplier_or_employee", "amount_ttc", "currency"], keep=False)
    return "No duplicate invoices (same supplier + amount)", inv[dup_mask]

This step defines ten automated control rules. Each rule inspects the data and returns the rows that fail it, for human review afterwards.

The rules check that: HT + VAT equals TTC; the VAT rate is legal (0 / 5.5 / 10 / 20%); invoices and expenses have a pre-tax amount; no amount is negative; no payment is dated before its document; the FX rate matches the currency; the euro conversion is correct; expenses stay under the 3,000 EUR limit; large invoices (over 30,000 EUR) have a purchase order; and no invoice is duplicated (same supplier, amount and currency).

Thresholds and reference values are grouped at the top so they can be adjusted in one place. This step produces no output — it only defines the rules, which are run in the next step.

In [9]:
# 4 — Run every control and produce a summary report
controls = [
    ctrl_ttc_consistency,
    ctrl_vat_rate_legal,
    ctrl_missing_amount,
    ctrl_negative_amount,
    ctrl_payment_after_invoice,
    ctrl_fx_consistency,
    ctrl_eur_conversion,
    ctrl_expense_policy,
    ctrl_large_invoice_has_po,
    ctrl_duplicate_invoices,
]

report_rows = []
for control in controls:
    label, flagged = control(wc)
    report_rows.append({
        "control": label,
        "status": "PASS" if len(flagged) == 0 else "FAIL",
        "flagged_rows": len(flagged),
    })

controls_report = pd.DataFrame(report_rows)
controls_report

,control,status,flagged_rows
0,TTC equals HT + VAT,FAIL,795
1,VAT rate is a legal rate,FAIL,360
2,Amount HT present on invoices/expenses,FAIL,210
3,No negative amounts,FAIL,140
4,Payment date not before document date,FAIL,250
5,FX rate matches currency,FAIL,160
6,amount_eur equals amount_ttc x fx_rate,FAIL,1356
7,Expense within policy limit,FAIL,180
8,Large invoice has a PO number,FAIL,6095
9,No duplicate invoices (same supplier + amount),FAIL,706


All ten controls returned a FAIL status, meaning each one found at least one row to review.

The controls flagged the following number of rows: 795 for TTC not equal to HT + VAT; 360 for a non-legal VAT rate; 210 for a missing HT amount on invoices or expenses; 140 for a negative amount; 250 for a payment dated before its document; 160 for an FX rate that does not match the currency; 1,356 for an incorrect euro conversion; 180 for an expense above the policy limit; 6,095 for a large invoice without a purchase order; and 706 for duplicate invoices.

In [10]:
# 5 — Collect every flagged row, labelled with the control it failed
flagged_details = []
for control in controls:
    label, flagged = control(wc)
    if len(flagged) > 0:
        detail = flagged.copy()
        detail.insert(0, "control_failed", label)
        flagged_details.append(detail)

anomalies_detail = pd.concat(flagged_details, ignore_index=True)

print("Total flagged rows (with overlaps):", len(anomalies_detail))
print("Unique documents flagged:", anomalies_detail["doc_id"].nunique())
print()
anomalies_detail[["control_failed", "doc_id", "doc_type", "supplier_or_employee",
                  "amount_ttc", "currency", "vat_rate", "po_number"]]

Total flagged rows (with overlaps): 10252
Unique documents flagged: 8627



,control_failed,doc_id,doc_type,supplier_or_employee,amount_ttc,currency,vat_rate,po_number
0,TTC equals HT + VAT,INV-185433,supplier_invoice,Keysight Technologies,12737.95,USD,0.0,PO-268688
1,TTC equals HT + VAT,INV-159949,supplier_invoice,Xometry Europe,31015.32,EUR,0.2,PO-222175
2,TTC equals HT + VAT,INV-168328,supplier_invoice,DHL France,52493.51,EUR,0.2,PO-250072
3,TTC equals HT + VAT,INV-110057,supplier_invoice,DHL France,-13709.93,EUR,0.2,PO-209362
4,TTC equals HT + VAT,INV-109852,supplier_invoice,Dassault Systemes,7148.20,EUR,0.2,PO-256134
...,...,...,...,...,...,...,...,...
10247,No duplicate invoices (same supplier + amount),INV-990031,supplier_invoice,Novanta Inc,27357.39,USD,0.0,NaN
10248,No duplicate invoices (same supplier + amount),INV-162775,supplier_invoice,Rohde & Schwarz,8918.77,EUR,0.2,PO-286404
10249,No duplicate invoices (same supplier + amount),INV-159827,supplier_invoice,Cabinet Juridique Lefevre,45926.03,EUR,0.2,PO-292591
10250,No duplicate invoices (same supplier + amount),INV-193482,supplier_invoice,Mouser Electronics,4299.81,USD,0.0,PO-259286


This step gathers every flagged row into a single table, with a first column indicating which control each row failed.

The table contains 10,252 flagged rows in total, counting overlaps (a document that fails several controls appears once per control).

These correspond to 8,627 unique documents, after removing those counted more than once.

Each row shows the failed control, the document id, its type, the supplier or employee, the total amount, the currency, the VAT rate and the purchase order number, so that each item can be reviewed individually.

In [11]:
# 6 — Export a multi-sheet Excel report for the finance team
import re

output_path = "/content/wandercraft_anomaly_report.xlsx"

def safe_sheet_name(name):
    # Remove characters Excel forbids in sheet names, then cap at 31 chars
    cleaned = re.sub(r'[/\\?*\[\]:]', '', name)
    return cleaned[:31]

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    # Sheet 1: summary (one row per control)
    controls_report.to_excel(writer, sheet_name="Summary", index=False)

    # Sheet 2: full detail of every flagged row
    export_cols = [
        "control_failed", "doc_id", "doc_type", "doc_date", "supplier_or_employee",
        "country", "category", "currency", "amount_ht", "vat_rate", "vat_amount",
        "amount_ttc", "fx_rate", "amount_eur", "payment_date", "po_number", "status",
    ]
    anomalies_detail[export_cols].to_excel(writer, sheet_name="All anomalies", index=False)

    # One sheet per failing control, for focused review
    for control in controls:
        label, flagged = control(wc)
        if len(flagged) > 0:
            flagged.to_excel(writer, sheet_name=safe_sheet_name(label), index=False)

print("Excel report written to:", output_path)

Excel report written to: /content/wandercraft_anomaly_report.xlsx


This step writes the anomaly report to an Excel file, so the finance team can open and work with the results directly in their usual tool.

The file contains one "Summary" sheet listing the ten controls and their status, one "All anomalies" sheet with the full detail of every flagged row, and one additional sheet per failing control containing only the rows it flagged, for focused review.

In [12]:
# 7 — A simple agent that reviews one document and writes a natural-language verdict
def review_document(doc):
    issues = []

    # Arithmetic consistency
    if pd.notna(doc["amount_ht"]) and pd.notna(doc["vat_amount"]):
        if abs(doc["amount_ht"] + doc["vat_amount"] - doc["amount_ttc"]) > 0.01:
            issues.append("the total (TTC) does not equal HT + VAT")

    # Legal VAT rate
    if doc["doc_type"] in ["supplier_invoice", "expense_report"] and pd.notna(doc["vat_rate"]):
        if round(doc["vat_rate"], 3) not in [round(r, 3) for r in LEGAL_VAT_RATES]:
            issues.append(f"the VAT rate ({doc['vat_rate']:.4f}) is not a legal rate")

    # Missing HT
    if doc["doc_type"] in ["supplier_invoice", "expense_report"] and pd.isna(doc["amount_ht"]):
        issues.append("the pre-tax amount (HT) is missing")

    # Negative amount
    if doc["amount_ttc"] < 0:
        issues.append("the amount is negative")

    # Payment chronology
    if pd.to_datetime(doc["payment_date"]) < pd.to_datetime(doc["doc_date"]):
        issues.append("the payment date is before the document date")

    # FX consistency
    expected_fx = EXPECTED_FX.get(doc["currency"])
    if expected_fx is not None and abs(doc["fx_rate"] - expected_fx) > 0.001:
        issues.append(f"the FX rate ({doc['fx_rate']}) does not match the currency ({doc['currency']})")

    # EUR conversion
    if abs(doc["amount_ttc"] * doc["fx_rate"] - doc["amount_eur"]) > 0.5:
        issues.append("the euro conversion is incorrect")

    # Expense policy
    if doc["doc_type"] == "expense_report" and pd.notna(doc["amount_ht"]) and doc["amount_ht"] > POLICY_LIMIT:
        issues.append(f"the expense exceeds the policy limit of {POLICY_LIMIT} EUR")

    # Large invoice without PO
    if doc["doc_type"] == "supplier_invoice" and abs(doc["amount_ttc"]) > LARGE_INVOICE and str(doc["po_number"]).strip() == "":
        issues.append(f"this large invoice (> {LARGE_INVOICE} EUR) has no purchase order")

    # Build the verdict
    if not issues:
        return f"[OK] {doc['doc_id']} ({doc['supplier_or_employee']}): no anomaly detected."
    else:
        problems = "; ".join(issues)
        return (f"[REVIEW] {doc['doc_id']} ({doc['supplier_or_employee']}, "
                f"{doc['amount_ttc']} {doc['currency']}): {len(issues)} issue(s) found — {problems}. "
                f"Recommended action: hold and review before payment.")

# Test the agent on a few documents
for i in [0, 1, 2, 3, 4]:
    print(review_document(wc.iloc[i]))
    print()

[OK] INV-117538 (TUV Rheinland): no anomaly detected.

[OK] BNK-5020385 (FedEx Express): no anomaly detected.

[OK] INV-117848 (Keysight Technologies): no anomaly detected.

[OK] BNK-5014007 (Mouser Electronics): no anomaly detected.

[OK] EXP-705487 (ROUX Camille): no anomaly detected.



This step defines an agent that reviews one document at a time and writes a plain-language verdict, the way an assistant accountant would annotate an invoice.

For each document, the agent applies all the control rules, collects the problems it finds, and returns a short verdict. A clean document gets an [OK] verdict; a document with issues gets a [REVIEW] verdict that lists every problem found and recommends holding the item for review before payment.

The agent never blocks or pays anything on its own: it explains and recommends, leaving the final decision to a human ("AI proposes, human validates").

Here the agent is tested on the first five documents of the dataset, which all return [OK] (no anomaly detected).

In [13]:
# 8 — Run the agent on a sample of documents known to have issues
sample_flagged = anomalies_detail["doc_id"].drop_duplicates().head(10)

for doc_id in sample_flagged:
    doc = wc[wc["doc_id"] == doc_id].iloc[0]
    print(review_document(doc))
    print()

[REVIEW] INV-185433 (Keysight Technologies, 12737.95 USD): 2 issue(s) found — the total (TTC) does not equal HT + VAT; the euro conversion is incorrect. Recommended action: hold and review before payment.

[REVIEW] INV-159949 (Xometry Europe, 31015.32 EUR): 1 issue(s) found — the pre-tax amount (HT) is missing. Recommended action: hold and review before payment.

[REVIEW] INV-168328 (DHL France, 52493.51 EUR): 2 issue(s) found — the total (TTC) does not equal HT + VAT; the euro conversion is incorrect. Recommended action: hold and review before payment.

[REVIEW] INV-110057 (DHL France, -13709.93 EUR): 3 issue(s) found — the total (TTC) does not equal HT + VAT; the amount is negative; the euro conversion is incorrect. Recommended action: hold and review before payment.

[REVIEW] INV-109852 (Dassault Systemes, 7148.2 EUR): 2 issue(s) found — the total (TTC) does not equal HT + VAT; the euro conversion is incorrect. Recommended action: hold and review before payment.

[REVIEW] INV-156620

This step runs the agent on documents already known to contain issues, to illustrate its output on real problem cases.

For readability, only the first 20 flagged documents are shown here.

Each verdict is a [REVIEW] describing the specific anomalies of that document and recommending that it be held for review before payment. Some documents carry several problems at once, which the agent combines into a single readable sentence.

As before, the agent only explains and recommends: the final decision is left to a human ("AI proposes, human validates").

In [14]:
# 9 — Install the Gemini library and load the API key from Colab secrets
!pip install -q google-generativeai

import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))

print("Gemini configured.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Gemini configured.


This step connects the notebook to Google's Gemini language model, which will be used to write the anomaly verdicts in natural language.

In [15]:
# 10 — An agent that asks Gemini to review a document and write a verdict
model = genai.GenerativeModel("models/gemini-flash-latest")

def review_document_llm(doc):
    # Build a compact, factual description of the document for the model
    facts = (
        f"Document ID: {doc['doc_id']}\n"
        f"Type: {doc['doc_type']}\n"
        f"Supplier/employee: {doc['supplier_or_employee']}\n"
        f"Document date: {doc['doc_date']} | Payment date: {doc['payment_date']}\n"
        f"Currency: {doc['currency']} | FX rate: {doc['fx_rate']}\n"
        f"Amount HT: {doc['amount_ht']} | VAT rate: {doc['vat_rate']} | "
        f"VAT amount: {doc['vat_amount']} | Amount TTC: {doc['amount_ttc']} | "
        f"Amount EUR: {doc['amount_eur']}\n"
        f"PO number: {doc['po_number']}"
    )

    prompt = (
        "You are an assistant accountant performing a first-level control on a single "
        "accounting document. Review the document below and identify any anomalies "
        "(arithmetic errors, non-standard VAT rate, missing data, negative amount, "
        "payment before invoice date, FX or currency-conversion errors, missing purchase "
        "order on a large invoice, or anything suspicious). "
        "Reply in 2-3 sentences: state whether the document is OK or needs review, list the "
        "issues found, and recommend an action. Do not invent data.\n\n"
        f"{facts}"
    )

    response = model.generate_content(prompt)
    return response.text.strip()

# Test on one clean document and one flagged document
clean_doc = wc[wc["doc_id"] == "INV-117538"].iloc[0]
flagged_id = anomalies_detail["doc_id"].iloc[0]
flagged_doc = wc[wc["doc_id"] == flagged_id].iloc[0]

print("CLEAN DOCUMENT:")
print(review_document_llm(clean_doc))
print()
print("FLAGGED DOCUMENT:")
print(review_document_llm(flagged_doc))

CLEAN DOCUMENT:
This document is OK for processing as no anomalies, arithmetic errors, or missing data were detected. All calculations are correct (VAT at 20% on €4,915.60 equals €983.12, totaling €5,898.72), the payment date occurs after the invoice date, and a purchase order number is provided. I recommend approving this invoice for standard payment.

FLAGGED DOCUMENT:
This document needs review due to an arithmetic error where the Amount TTC (12,737.95 USD) does not match the sum of the Amount HT (12,366.90 USD) and the VAT amount (0.00 USD). Furthermore, there is an FX conversion discrepancy, as the Amount EUR (11,377.55 EUR) was calculated using the Amount HT instead of the stated Amount TTC. I recommend contacting the supplier to clarify these calculation discrepancies and request a corrected invoice.


In [16]:
# 11 — Tools the agent can use to act on a document
# Each tool performs one concrete action and returns a short record of what it did.

def tool_approve(doc):
    return {
        "action": "APPROVED",
        "doc_id": doc["doc_id"],
        "detail": "No anomaly detected. Document approved for standard posting and payment.",
        "email_draft": None,
    }

def tool_hold(doc, issues):
    return {
        "action": "ON HOLD",
        "doc_id": doc["doc_id"],
        "detail": f"Minor issue(s) found: {issues}. Document put on hold for accountant review.",
        "email_draft": None,
    }

def tool_block_and_draft_email(doc, issues):
    # For serious issues: block the document and let the LLM draft a supplier email
    prompt = (
        "You are an assistant accountant. Write a short, polite email (4-5 sentences) to a "
        "supplier to report a problem with their invoice and request a corrected version. "
        "Be specific about the issue. Do not invent data.\n\n"
        f"Supplier: {doc['supplier_or_employee']}\n"
        f"Invoice: {doc['doc_id']}\n"
        f"Amount: {doc['amount_ttc']} {doc['currency']}\n"
        f"Problem detected: {issues}"
    )
    try:
        email = model.generate_content(prompt).text.strip()
    except Exception as e:
        email = f"(email draft could not be generated: {e})"

    return {
        "action": "BLOCKED",
        "doc_id": doc["doc_id"],
        "detail": f"Serious issue(s) found: {issues}. Document blocked, pending human validation.",
        "email_draft": email,
    }

print("Tools defined: approve, hold, block_and_draft_email.")

Tools defined: approve, hold, block_and_draft_email.


This step defines the three tools the agent can use to act on a document. A tool is a small function that performs one concrete action and returns a record of what it did.

The approve tool marks a clean document as approved for payment. The hold tool puts a document with a minor issue on hold for the accountant to review. The block tool blocks a document with a serious issue and uses the language model to draft a corrective email to the supplier.

None of these tools performs an irreversible action: an approval does not actually pay anything, and a block only prepares a draft email. A human validates before anything is really sent or paid.

This step only defines the tools; the agent that chooses between them is built in the next step.

In [17]:
# 12 — The agent inspects a document, decides on a severity, and calls the right tool

SERIOUS_ISSUES = {"negative amount", "missing HT", "large invoice without PO", "duplicate invoice"}

def analyse_issues(doc):
    # Reuse the control logic to list the issues found on this document
    issues = []
    if doc["amount_ttc"] < 0:
        issues.append("negative amount")
    if doc["doc_type"] in ["supplier_invoice", "expense_report"] and pd.isna(doc["amount_ht"]):
        issues.append("missing HT")
    if doc["doc_type"] == "supplier_invoice" and abs(doc["amount_ttc"]) > LARGE_INVOICE and str(doc["po_number"]).strip() == "":
        issues.append("large invoice without PO")
    if pd.notna(doc["amount_ht"]) and pd.notna(doc["vat_amount"]):
        if abs(doc["amount_ht"] + doc["vat_amount"] - doc["amount_ttc"]) > 0.01:
            issues.append("TTC does not equal HT + VAT")
    if doc["doc_type"] in ["supplier_invoice", "expense_report"] and pd.notna(doc["vat_rate"]):
        if round(doc["vat_rate"], 3) not in [round(r, 3) for r in LEGAL_VAT_RATES]:
            issues.append("non-standard VAT rate")
    if pd.to_datetime(doc["payment_date"]) < pd.to_datetime(doc["doc_date"]):
        issues.append("payment before invoice date")
    expected_fx = EXPECTED_FX.get(doc["currency"])
    if expected_fx is not None and abs(doc["fx_rate"] - expected_fx) > 0.001:
        issues.append("FX rate inconsistent with currency")
    return issues

def agent_process(doc):
    issues = analyse_issues(doc)

    # DECISION: choose an action based on what was found
    if not issues:
        return tool_approve(doc)
    elif SERIOUS_ISSUES.intersection(issues):
        return tool_block_and_draft_email(doc, ", ".join(issues))
    else:
        return tool_hold(doc, ", ".join(issues))

# Test the agent on a clean document and a few flagged documents
flagged_id = anomalies_detail["doc_id"].iloc[0]
test_ids = ["INV-117538", flagged_id]

for tid in test_ids:
    doc = wc[wc["doc_id"] == tid].iloc[0]
    result = agent_process(doc)
    print(f"{result['action']} — {result['doc_id']}")
    print(result["detail"])
    if result["email_draft"]:
        print("--- Draft email ---")
        print(result["email_draft"])
    print()

APPROVED — INV-117538
No anomaly detected. Document approved for standard posting and payment.

ON HOLD — INV-185433
Minor issue(s) found: TTC does not equal HT + VAT. Document put on hold for accountant review.



The agent inspects a document, decides what to do, and acts. A clean document is approved, a minor issue is put on hold, and a serious issue (negative amount, missing HT, large invoice without PO, or duplicate) is blocked.

Here the first document (INV-117538) is clean and gets approved, while the second (INV-185433) has a minor issue — TTC does not equal HT + VAT — and is put on hold.

What makes this an agent is that it chooses between several actions and acts on that decision, rather than only giving an opinion.

In [23]:
# 12 (revised) — Agent with an optional email-drafting switch
def agent_process(doc, draft_email=True):
    issues = analyse_issues(doc)

    if not issues:
        return tool_approve(doc)
    elif SERIOUS_ISSUES.intersection(issues):
        if draft_email:
            return tool_block_and_draft_email(doc, ", ".join(issues))
        else:
            # Block without calling the LLM (for bulk processing)
            return {
                "action": "BLOCKED",
                "doc_id": doc["doc_id"],
                "detail": f"Serious issue(s) found: {', '.join(issues)}. Document blocked, pending human validation.",
                "email_draft": None,
            }
    else:
        return tool_hold(doc, ", ".join(issues))

# 13 (revised) — Process every document WITHOUT calling the LLM (bulk mode)
agent_results = []
for _, doc in wc.iterrows():
    result = agent_process(doc, draft_email=False)   # <-- no LLM call
    agent_results.append({
        "doc_id": result["doc_id"],
        "action": result["action"],
        "detail": result["detail"],
    })

agent_output = pd.DataFrame(agent_results)

print("Actions summary:")
print(agent_output["action"].value_counts())
print()
agent_output.head(20)

Actions summary:
action
APPROVED    136219
ON HOLD       1243
BLOCKED        350
Name: count, dtype: int64



,doc_id,action,detail
0,INV-117538,APPROVED,No anomaly detected. Document approved for sta...
1,BNK-5020385,APPROVED,No anomaly detected. Document approved for sta...
2,INV-117848,APPROVED,No anomaly detected. Document approved for sta...
3,BNK-5014007,APPROVED,No anomaly detected. Document approved for sta...
4,EXP-705487,APPROVED,No anomaly detected. Document approved for sta...
5,INV-138286,APPROVED,No anomaly detected. Document approved for sta...
6,INV-178469,APPROVED,No anomaly detected. Document approved for sta...
7,INV-114472,APPROVED,No anomaly detected. Document approved for sta...
8,INV-135865,APPROVED,No anomaly detected. Document approved for sta...
9,INV-184422,APPROVED,No anomaly detected. Document approved for sta...


This step runs the agent on every document in the file, then shows only the first 10 results as a preview.

Across the whole dataset, the agent sorted each document into one of three actions: approved, put on hold, and blocked.

To process the full file, the agent decides without calling the language model — email drafting would require one model call per blocked document and would exhaust the API quota. Email generation is therefore kept for the individual tests only.

This shows that the agent scales to the entire file, applying the same decision logic consistently across all 137,812 documents.

In [24]:
# 14 — Export all agent decisions (approved / on hold / blocked) to Excel
agent_export_path = "/content/agent_decisions.xlsx"

# Merge the agent decision back with the key document fields
agent_full = wc.merge(agent_output, on="doc_id", how="left")

export_cols = [
    "doc_id", "action", "detail", "doc_type", "supplier_or_employee",
    "amount_ttc", "currency", "amount_eur", "doc_date", "payment_date", "po_number",
]

with pd.ExcelWriter(agent_export_path, engine="openpyxl") as writer:
    # One sheet with everything
    agent_full[export_cols].to_excel(writer, sheet_name="All decisions", index=False)

    # One sheet per action type
    for action in ["APPROVED", "ON HOLD", "BLOCKED"]:
        subset = agent_full[agent_full["action"] == action][export_cols]
        writer_sheet = action.replace(" ", "_")
        subset.to_excel(writer, sheet_name=writer_sheet, index=False)

print("Agent decisions exported to:", agent_export_path)
print()
print(agent_full["action"].value_counts())

Agent decisions exported to: /content/agent_decisions.xlsx

action
APPROVED    136219
ON HOLD       1243
BLOCKED        350
Name: count, dtype: int64
